# 20_model_eval — 최종 학습데이터 모델 학습 & 성능 평가 (1:1)

최종 1:1 데이터로 **fingerprint만 / descriptor만 / 결합** 세 가지 특징 집합을 같은 조건
(5-fold 교차검증, RandomForest)에서 비교한다.

**성능 지표** (혼동행렬 TP/TN/FP/FN 기반):
- **MCC** (1): 불균형에도 견고한 균형 지표
- **Accuracy** (2) = (TP+TN)/전체
- **Recall** (3) = TP/(TP+FN) — 실제 양성을 얼마나 잡나
- **Precision** (4) = TP/(TP+FP) — 예측 양성 중 진짜 비율
- **ROC-AUC**, **PR-AUC**

> 모델은 QSAR 표준 RandomForest. `make_model()`만 바꾸면 다른 분류기로 교체 가능.

In [ ]:
# 프로젝트 루트로 이동 + import
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')            # 파일 저장용(디스플레이 없어도 동작). 노트북에선 자동 표시됨
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix,
                             matthews_corrcoef, roc_curve, precision_recall_curve)

### 데이터 로드 & 특징 집합 (fingerprint / descriptor / 결합)

In [ ]:
# 최종 학습데이터 로드 + 3가지 특징 집합 정의
SRC = 'data/HSD17B13_final_training_1to1.csv'
df = pd.read_csv(SRC)
y = df['potency'].to_numpy()

fp_cols = [c for c in df.columns if c.startswith('fp_')]
non_desc = set(['canonical_smiles', 'potency'] + fp_cols)
desc_cols = [c for c in df.columns if c not in non_desc]
print('화합물', len(df), '| potency', dict(pd.Series(y).value_counts()))
print('fingerprint 열', len(fp_cols), '| descriptor 열', len(desc_cols))

FEATURES = {
    'fingerprint':      df[fp_cols].to_numpy(),
    'descriptor':       df[desc_cols].to_numpy(),
    '결합(FP+desc)':    df[fp_cols + desc_cols].to_numpy(),
}

### 성능 지표 함수 (논문 정의대로 혼동행렬 기반)

In [ ]:
# 성능 지표 함수 (논문 정의: 혼동행렬 TP/TN/FP/FN 기반)
def metrics_from(y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    acc = (tp + tn) / (tp + tn + fp + fn)                 # (2) Accuracy
    rec = tp / (tp + fn) if (tp + fn) else 0.0            # (3) Recall
    prec = tp / (tp + fp) if (tp + fp) else 0.0           # (4) Precision
    mcc = matthews_corrcoef(y_true, pred)                 # (1) MCC
    roc = roc_auc_score(y_true, proba)                    # AUC (ROC)
    pr = average_precision_score(y_true, proba)           # PR-AUC
    return dict(ROC_AUC=roc, PR_AUC=pr, MCC=mcc, Accuracy=acc,
               Recall=rec, Precision=prec, TP=tp, TN=tn, FP=fp, FN=fn)

### 3종 비교 (5-fold CV)

In [ ]:
# 3종 특징 집합을 같은 조건(5-fold CV, RandomForest)으로 비교
def make_model():
    return RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                  random_state=42, n_jobs=-1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows, proba_store = [], {}
for name, X in FEATURES.items():
    proba = cross_val_predict(make_model(), X, y, cv=skf,
                              method='predict_proba', n_jobs=-1)[:, 1]
    proba_store[name] = proba
    m = metrics_from(y, proba)
    m = {'features': name, **m}
    rows.append(m)
    print('[%s] ROC-AUC %.3f | PR-AUC %.3f | MCC %.3f | Acc %.3f | Recall %.3f | Prec %.3f'
          % (name, m['ROC_AUC'], m['PR_AUC'], m['MCC'], m['Accuracy'], m['Recall'], m['Precision']))

res = pd.DataFrame(rows)
OUT = 'data/HSD17B13_model_eval_1to1.csv'
res.to_csv(OUT, index=False)
print('\n결과 저장:', OUT)
print(res[['features', 'ROC_AUC', 'PR_AUC', 'MCC', 'Accuracy', 'Recall', 'Precision']]
      .round(3).to_string(index=False))

### ROC / PR 커브

In [ ]:
# ROC / PR 커브 (3종 비교)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for name, proba in proba_store.items():
    fpr, tpr, _ = roc_curve(y, proba)
    ax[0].plot(fpr, tpr, label='%s (AUC=%.3f)' % (name, roc_auc_score(y, proba)))
    pr, rc, _ = precision_recall_curve(y, proba)
    ax[1].plot(rc, pr, label='%s (PR-AUC=%.3f)' % (name, average_precision_score(y, proba)))
ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
ax[0].set_title('ROC curve'); ax[0].legend(loc='lower right', fontsize=9)
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision')
ax[1].set_title('Precision-Recall curve'); ax[1].legend(loc='lower left', fontsize=9)
plt.tight_layout()
FIG = 'data/HSD17B13_model_eval_1to1.png'
plt.savefig(FIG, dpi=130)
plt.show()
print('그림 저장:', FIG)